In [2]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ["TOKENIZERS_PARALLELISM"] = "false"


In [3]:
#Load dataset from Huggingface
from datasets import load_dataset

dataset = load_dataset("ag_news")

In [4]:
#Check one sample
print(dataset['train'][0])

{'text': "Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\\band of ultra-cynics, are seeing green again.", 'label': 2}


In [5]:
#Subsample to reduce training time (critical on Kaggle)
dataset["train"] = dataset["train"].shuffle(seed=42).select(range(8000))
dataset["test"] = dataset["test"].shuffle(seed=42).select(range(2000))

In [6]:
#Transformers like BERT need text to be converted into tokens before being passed to the model
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True)
tokenized_dataset = dataset.map(tokenize, batched=True)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [7]:
#Data Collator
from transformers import DataCollatorWithPadding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

2025-07-26 20:14:37.881527: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753560878.050048      65 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753560878.103082      65 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [8]:
#We load the pre-trained BERT model for classification and define training parameters.
from transformers import BertForSequenceClassification
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=4)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [9]:
#Training Arguments
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    weight_decay=0.01,
    fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    save_total_limit=1,
    report_to="none",
    gradient_accumulation_steps=2,
    dataloader_num_workers=2,
)

In [10]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    precision, recall, f1, _ = precision_recall_fscore_support(labels, preds, average="weighted")
    acc = accuracy_score(labels, preds)
    return {"accuracy": acc, "f1": f1, "precision": precision, "recall": recall}

In [11]:
#Setup Trainer
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [12]:
#Train the model
trainer.train()

#Evaluate the model
trainer.evaluate()

/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy,F1,Precision,Recall
1,0.311100,0.304243,0.902000,0.901777,0.904423,0.902000
2,0.186700,0.264487,0.915000,0.914904,0.916345,0.915000


/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/torch/nn/parallel/_functions.py:70: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.2644865810871124,
 'eval_accuracy': 0.915,
 'eval_f1': 0.914904180587079,
 'eval_precision': 0.9163450853317688,
 'eval_recall': 0.915,
 'eval_runtime': 9.0599,
 'eval_samples_per_second': 220.754,
 'eval_steps_per_second': 6.954,
 'epoch': 2.0}

In [13]:
#Save model + tokenizer for deployment
save_path = "./news-bert-model"
trainer.save_model(save_path)
tokenizer.save_pretrained(save_path)

('./news-bert-model/tokenizer_config.json',
 './news-bert-model/special_tokens_map.json',
 './news-bert-model/vocab.txt',
 './news-bert-model/added_tokens.json',
 './news-bert-model/tokenizer.json')

In [16]:
!zip -r news-bert-model-v1.zip news-bert-model


  adding: news-bert-model/ (stored 0%)
  adding: news-bert-model/special_tokens_map.json (deflated 42%)
  adding: news-bert-model/config.json (deflated 52%)
  adding: news-bert-model/tokenizer_config.json (deflated 75%)
  adding: news-bert-model/tokenizer.json (deflated 71%)
  adding: news-bert-model/training_args.bin (deflated 51%)
  adding: news-bert-model/model.safetensors (deflated 7%)
  adding: news-bert-model/vocab.txt (deflated 53%)
